# Preprocess Dataset for 2.5D Experiments
#
Reads the Kaggle-processed lung tumour segmentation dataset (which was
normalised by dividing raw HU values by 3071), **recovers the original
Hounsfield Units**, applies proper lung-window clipping, and re-normalises
to [0, 1].
#
Also re-splits patients into train / val / test, preserves **all** slices
in anatomical order (needed for 2.5D triplet loading), and writes
manifest JSONs with balanced sampling indices.

In [1]:
import json
import os
import random
import shutil

import numpy as np

In [2]:
# ----- paths -----
# On Kaggle the original dataset is mounted under /kaggle/input/...
# Adjust `original_root` to wherever the unprocessed data lives.
_kaggle_root = '/kaggle/input/datasets/rasoulisaeid/lung-cancer-segment'
_local_root = os.path.join(os.path.dirname(os.path.abspath('.')), 'Lung Tumor Segmentation')
original_root = _kaggle_root if os.path.isdir(_kaggle_root) else _local_root

original_train_dir = os.path.join(original_root, 'train')
original_val_dir = os.path.join(original_root, 'val')

output_root = os.path.join(os.getcwd(), 'lung_tumor_processed')
manifest_dir = os.path.join(output_root, 'manifests')
os.makedirs(manifest_dir, exist_ok=True)

# ---- Re-normalisation parameters ----
# The Kaggle dataset was normalised as: npy_value = raw_hu / 3071
# We recover HU, clip to a lung window, and scale to [0, 1].
KAGGLE_DIVISOR = 3071.0
HU_MIN = -1024.0
HU_MAX = 600.0


def renormalize_slice(arr: np.ndarray) -> np.ndarray:
    """Undo the /3071 normalisation, apply lung-window clipping, rescale to [0, 1]."""
    hu = arr.astype(np.float32) * KAGGLE_DIVISOR
    clipped = np.clip(hu, HU_MIN, HU_MAX)
    return ((clipped - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float32)


print(f'Original root : {original_root}')
print(f'Output root   : {output_root}')
print(f'HU window     : [{HU_MIN}, {HU_MAX}]')

Original root : /kaggle/input/datasets/rasoulisaeid/lung-cancer-segment
Output root   : /kaggle/working/lung_tumor_processed
HU window     : [-1024.0, 600.0]


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SKIP_PATIENTS = {56}

SPLITS = [
    ('train', list(range(0, 52)), original_train_dir),
    ('val',   list(range(52, 56)), original_train_dir),
    ('test',  list(range(57, 63)), original_val_dir),
]

In [4]:
for split_name, patient_ids, source_dir in SPLITS:
    split_out = os.path.join(output_root, split_name)
    os.makedirs(split_out, exist_ok=True)

    samples = []

    for pid in patient_ids:
        if pid in SKIP_PATIENTS:
            continue

        src_patient = os.path.join(source_dir, str(pid))
        src_data = os.path.join(src_patient, 'data')
        src_masks = os.path.join(src_patient, 'masks')

        if not os.path.isdir(src_data):
            print(f'  [WARN] Missing data dir for patient {pid}: {src_data}')
            continue

        dst_patient = os.path.join(split_out, str(pid))
        dst_data = os.path.join(dst_patient, 'data')
        dst_masks = os.path.join(dst_patient, 'masks')
        os.makedirs(dst_data, exist_ok=True)
        os.makedirs(dst_masks, exist_ok=True)

        slice_files = sorted(
            [f for f in os.listdir(src_data) if f.endswith('.npy')],
            key=lambda f: int(os.path.splitext(f)[0]),
        )
        num_slices = len(slice_files)

        for fname in slice_files:
            raw = np.load(os.path.join(src_data, fname))
            np.save(os.path.join(dst_data, fname), renormalize_slice(raw))

            mask_src = os.path.join(src_masks, fname)
            if os.path.isfile(mask_src):
                shutil.copy2(mask_src, os.path.join(dst_masks, fname))
            else:
                print(f'  [WARN] Missing mask for patient {pid} slice {fname}')

        for idx, fname in enumerate(slice_files):
            mask_path = os.path.join(dst_masks, fname)
            has_tumor = False
            if os.path.isfile(mask_path):
                mask = np.load(mask_path)
                has_tumor = bool(mask.max() > 0)
            samples.append({
                'patient': str(pid),
                'slice_idx': idx,
                'num_slices': num_slices,
                'has_tumor': has_tumor,
            })

    tumor_indices = [i for i, s in enumerate(samples) if s['has_tumor']]
    non_tumor_indices = [i for i, s in enumerate(samples) if not s['has_tumor']]

    manifest = {
        'split': split_name,
        'patients': [str(pid) for pid in patient_ids if pid not in SKIP_PATIENTS],
        'samples': samples,
        'tumor_indices': tumor_indices,
        'non_tumor_indices': non_tumor_indices,
    }

    if split_name != 'test':
        n_tumor = len(tumor_indices)
        sampled_non_tumor = random.sample(non_tumor_indices, min(n_tumor, len(non_tumor_indices)))
        balanced = tumor_indices + sampled_non_tumor
        random.shuffle(balanced)
        manifest['balanced_indices'] = balanced

    manifest_path = os.path.join(manifest_dir, f'{split_name}_manifest.json')
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    n_bal = len(manifest.get('balanced_indices', []))
    print(
        f'{split_name:>5}: {len(samples)} slices | '
        f'{len(tumor_indices)} tumor | {len(non_tumor_indices)} non-tumor | '
        f'balanced={n_bal}'
    )

print('\nDone. Manifests written to', manifest_dir)

train: 13470 slices | 1451 tumor | 12019 non-tumor | balanced=2902
  val: 877 slices | 87 tumor | 790 non-tumor | balanced=174
 test: 1335 slices | 101 tumor | 1234 non-tumor | balanced=0

Done. Manifests written to /kaggle/working/lung_tumor_processed/manifests
